# Generation — Answer from Context

## Goal
Use retrieved chunks as context to generate an answer.
This is the "G" in RAG — Retrieval-Augmented Generation.

## Approach
Model: translategemma:4b — open source, runs locally via Ollama.

## Pipeline
Query -> Retrieve Chunks -> Build Prompt -> Generate Answer

In [13]:
import re
import numpy as np
import faiss
import requests
from sentence_transformers import SentenceTransformer

## 1. Load Models and Build Knowledge Base

In [14]:
print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Done!")

sample_document = """
Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve from experience without being explicitly programmed. 
It focuses on developing computer programs that can access data and use it 
to learn for themselves.

The process begins with observations or data, such as examples, direct 
experience, or instruction. Machine learning algorithms use computational 
methods to learn information directly from data without relying on a 
predetermined equation as a model.

Deep learning is a subfield of machine learning that uses neural networks 
with many layers. These deep neural networks have revolutionized fields such 
as computer vision, natural language processing, and speech recognition.

Natural language processing (NLP) is another important area of machine learning. 
It enables computers to understand, interpret, and generate human language. 
Applications include sentiment analysis, machine translation, and chatbots.

Reinforcement learning is a type of machine learning where an agent learns 
to make decisions by interacting with an environment. The agent receives 
rewards for correct actions and penalties for incorrect ones.
"""

def clean_text(text: str) -> str:
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r' +', ' ', text)
    return text.strip()

def chunk_by_sentences(text: str, sentences_per_chunk: int = 3) -> list:
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i + sentences_per_chunk])
        if chunk.strip():
            chunks.append(chunk.strip())
    return chunks

cleaned_doc = clean_text(sample_document)
chunks = chunk_by_sentences(cleaned_doc)
chunk_embeddings = embed_model.encode(chunks).astype(np.float32)
index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

print(f"Knowledge base ready. Chunks: {len(chunks)}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Done!
Knowledge base ready. Chunks: 4


## 2. RAG Pipeline

In [15]:
def retrieve(query: str, top_k: int = 2) -> list:
    """Retrieve top-k relevant chunks for a query."""
    query_embedding = embed_model.encode([query]).astype(np.float32)
    distances, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0]]


def generate(query: str, context: list) -> str:
    """Generate answer using Ollama via direct HTTP request."""
    context_text = '\n'.join(context)
    
    prompt = f"""Answer the question based only on the context below.
Context:
{context_text}

Question: {query}
Answer:"""
    
    response = requests.post(
        'http://127.0.0.1:11434/api/chat',
        json={
            'model': 'translategemma:4b',
            'messages': [{'role': 'user', 'content': prompt}],
            'stream': False
        },
        proxies={'http': None, 'https': None}
    )
    return response.json()['message']['content']


def rag(query: str) -> dict:
    """Full RAG pipeline: retrieve + generate."""
    context = retrieve(query)
    answer = generate(query, context)
    return {
        'query': query,
        'context': context,
        'answer': answer
    }

print("RAG pipeline ready.")

RAG pipeline ready.


## 3. Test RAG Pipeline

In [16]:
queries = [
    "What is deep learning?",
    "How does reinforcement learning work?",
    "What are NLP applications?",
]

for query in queries:
    print(f"\nQ: {query}")
    print("-" * 50)
    result = rag(query)
    print(f"A: {result['answer']}")
    print(f"\nContext used:")
    for i, ctx in enumerate(result['context']):
        print(f"  [{i+1}] {ctx[:100]}...")


Q: What is deep learning?
--------------------------------------------------
A: Deep learning is a subfield of machine learning that uses neural networks with many layers.


Context used:
  [1] Machine learning algorithms use computational 
methods to learn information directly from data witho...
  [2] Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve f...

Q: How does reinforcement learning work?
--------------------------------------------------
A: Reinforcement learning is a type of machine learning where an agent learns to make decisions by interacting with an environment. The agent receives rewards for correct actions and penalties for incorrect ones.


Context used:
  [1] Reinforcement learning is a type of machine learning where an agent learns 
to make decisions by int...
  [2] Machine learning is a subset of artificial intelligence that enables systems 
to learn and improve f...

Q: What are NLP applications?
------------------

## 4. Key Observations

| Query | Answer Quality | Context Relevant? |
|-------|---------------|-------------------|
| Deep learning | Accurate | Yes |
| Reinforcement learning | Accurate | Yes |
| NLP applications | Accurate, concise | Yes |

## Key Insight
RAG grounds the model in real documents.
The model cannot hallucinate facts it doesn't have —
it can only answer from the provided context.

## Limitation
If the answer is not in the document, the model may still try to answer.
This is why evaluation (hallucination analysis) is the next step.

## Next
-> 04_evaluation.ipynb